In [19]:
import pandas as pd
from geopy.distance import great_circle
import numpy as np
df = pd.read_csv("../data/raw/fraudTrain.csv")
pd.set_option('display.max_columns', None)

In [20]:
def clean_data(df:pd.DataFrame):
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["date_of_birth"] = pd.to_datetime(df["dob"])
    
    df.drop(columns=["Unnamed: 0","dob"],inplace=True)
    return df
    
    
def add_time(df:pd.DataFrame):
    df["week_day"] = df["trans_date_trans_time"].dt.day_name()
    df["hour"] = df["trans_date_trans_time"].dt.hour
    return df

    
def add_age(df:pd.DataFrame):
    df["age"] = df["trans_date_trans_time"].dt.year - df["date_of_birth"].dt.year
    bins = [0, 17, 24, 34, 49, 64, 100]
    labels = ["0-17", "18-24", "25-34","35-49", "50-64","65+"]
    df['age_group'] = pd.cut(df['age'],right=True, bins=bins, labels=labels, include_lowest=True)
    return df


def add_city_group(df:pd.DataFrame):
    bins = [0, 1000, 10000, 100000,500000, 1000000, 10000000]
    labels = ["rural","small_town","small_city","medium_city","large_city","mega_city"]
    df['city_group'] = pd.cut(df['city_pop'],right=True, bins=bins, labels=labels, include_lowest=True)
    return df


def calc_distance(a_lat:float, a_long:float, b_lat:float, b_long:float):
    point_a = (a_lat, a_long)
    point_b = (b_lat, b_long)
    return great_circle(point_a, point_b).km
    


def add_distance(df:pd.DataFrame):
    df["distance"] = df.apply(lambda row: calc_distance(row["lat"], row["long"], row["merch_lat"], row["merch_long"]), axis=1)
    return df


def add_distance_bucket(df:pd.DataFrame):
    bins = [0, 1, 10, 20, 100, 1000, 10000]
    labels = ["district","city","city_suburbs","metropolitan_area","country","world"]
    df['distance_group'] = pd.cut(df['distance'],right=True, bins=bins, labels=labels, include_lowest=True)
    return df


def add_transaction_avg(df:pd.DataFrame):
    average_amt_per_user_df = df.groupby(["cc_num"])["amt"].mean().reset_index()
    average_amt_per_user_df.rename(columns={"amt":"mean_amt"}, inplace=True)

    df = df.merge(average_amt_per_user_df, on="cc_num", how="left")
    return df


    
def add_amt_std(df:pd.DataFrame):
    user_amt_std_df = df.groupby(["cc_num"])["amt"].std().reset_index()
    user_amt_std_df.rename(columns={"amt":"amt_std"},inplace=True)

    df = df.merge(user_amt_std_df, on="cc_num", how="left")
    return df


def add_time_since(df:pd.DataFrame):
    df = df.sort_values(by=["cc_num", "trans_date_trans_time"])
    df["time_since_prev_transaction"] = (df.groupby("cc_num")["trans_date_trans_time"].diff())
    return df

def add_prev_transaction_distance(df):
    
    df = df.sort_values(by=["cc_num", "trans_date_trans_time"])

    # get previous merchant coordinates per user
    df["prev_merch_lat"] = df.groupby("cc_num")["merch_lat"].shift(1)
    df["prev_merch_long"] = df.groupby("cc_num")["merch_long"].shift(1)

    # compute distance between current and previous merchant location
    df["distance_from_prev"] = df.apply(
        lambda row: calc_distance(
            row["prev_merch_lat"],
            row["prev_merch_long"],
            row["merch_lat"],
            row["merch_long"]
        ) if not np.isnan(row["prev_merch_lat"]) else 0,
        axis=1
    )

    # cleanup
    df.drop(columns=["prev_merch_lat", "prev_merch_long"], inplace=True)

    return df


In [ ]:


df = clean_data(df)
df = add_time(df)
df = add_age(df)
df = add_city_group(df)
df = add_distance(df)
df = add_distance_bucket(df)
df = add_transaction_avg(df)
df = add_amt_std(df)
df = add_time_since(df)
df = add_prev_transaction_distance(df)
# df.to_csv("../data/processed/cleeaned_fraud_train.csv", index=False)

In [ ]:
# 7. Velocity Features

# For each transaction:

# number of transactions in last:
# 1 hour
# 24 hours

# Transaction Behavior per User


# assign day/night
# assign weekday/weekend

# distance from previous transaction

In [ ]:
df


,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,unix_time,merch_lat,merch_long,is_fraud,date_of_birth,week_day,hour,age,age_group,city_group,distance,distance_group,mean_amt,amt_std,time_since_prev_transaction
1017,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,98e3dcf98101146a577f85a34e58feec,1325422035,43.974711,-109.741904,0,1986-02-17,Tuesday,12,33,25-34,small_town,127.606419,country,56.023366,122.632635,NaT
2724,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,498120fc45d277f7c88e3dba79c33865,1325493897,42.018766,-109.044172,0,1986-02-17,Wednesday,8,33,25-34,small_town,110.309077,country,56.023366,122.632635,0 days 19:57:42
2726,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,95f514bb993151347c7acdf8505c3d62,1325494056,42.961335,-109.157564,0,1986-02-17,Wednesday,8,33,25-34,small_town,21.787292,metropolitan_area,56.023366,122.632635,0 days 00:02:39
2882,2019-01-02 12:38:14,60416207185,fraud_Daugherty LLC,kids_pets,34.79,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,4f0c1a14e0aa7eb56a490780ef9268c5,1325507894,42.228227,-108.747683,0,1986-02-17,Wednesday,12,33,25-34,small_town,87.204338,metropolitan_area,56.023366,122.632635,0 days 03:50:38
2907,2019-01-02 13:10:46,60416207185,fraud_Beier and Sons,home,27.18,Mary,Diaz,F,9886 Anita Drive,Fort Washakie,WY,82514,43.0048,-108.8964,1645,Information systems manager,3b2ebd3af508afba959640893e1e82bc,1325509846,43.321745,-108.091143,0,1986-02-17,Wednesday,13,33,25-34,small_town,74.213070,metropolitan_area,56.023366,122.632635,0 days 00:32:32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294934,2020-06-20 21:04:59,4992346398065154184,"fraud_Berge, Kautzer and Harris",personal_care,60.47,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,ad7dfdf0aaa36cd7985dd1f35ca0e2fc,1371762299,40.475395,-89.076105,0,1956-01-09,Saturday,21,64,50-64,rural,78.492673,metropolitan_area,67.843832,157.223610,0 days 08:32:20
1295369,2020-06-21 00:41:01,4992346398065154184,fraud_Bernhard Inc,gas_transport,74.29,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,6d427d735c9f9b2fd480f2c24b6525de,1371775261,40.743634,-89.553379,0,1956-01-09,Sunday,0,64,50-64,rural,55.400846,metropolitan_area,67.843832,157.223610,0 days 03:36:02
1295587,2020-06-21 02:47:59,4992346398065154184,"fraud_Reichert, Rowe and Mraz",shopping_net,246.56,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,9814049bcc69fb31d81f4a907f2fe255,1371782879,40.215418,-88.682562,0,1956-01-09,Sunday,2,64,50-64,rural,115.674563,country,67.843832,157.223610,0 days 02:06:58
1296206,2020-06-21 08:04:28,4992346398065154184,fraud_Jewess LLC,shopping_pos,2.62,Benjamin,Kim,M,920 Patrick Light,Mc Nabb,IL,61335,41.1730,-89.2187,532,Audiological scientist,ae39b91cd2b4897ddbbf6bf63d3e7b03,1371801868,40.762861,-88.744967,0,1956-01-09,Sunday,8,64,50-64,rural,60.513482,metropolitan_area,67.843832,157.223610,0 days 05:16:29
